# 05. Missing Value Analysis — Mechanisms, Diagnostics & Imputation

Missing data is not merely an inconvenience; it can introduce severe bias into machine learning estimators if handled improperly.

---

### Donald Rubin's Taxonomy of Missingness
1. **MCAR (Missing Completely at Random)**: The probability of missingness is unrelated to both observed and unobserved data. (e.g., A sensor battery randomly dies).
2. **MAR (Missing at Random)**: Missingness depends systematically on observed data, but not on the missing value itself. (e.g., Young men are less likely to report income in a survey, but conditional on age/gender, missingness is random).
3. **MNAR (Missing Not at Random)**: Missingness depends directly on the unobserved value itself. (e.g., High-wealth individuals refuse to disclose high incomes).

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer

np.random.seed(42)
n = 1000

# Generate baseline feature distributions
age = np.random.normal(45, 12, size=n).round(1)
income = (age * 1200 + np.random.normal(15000, 8000, size=n)).round(2)
credit_score = np.random.normal(680, 75, size=n).round(0)

df_missing = pd.DataFrame({'age': age, 'income': income, 'credit_score': credit_score})

# Inject MCAR missingness into age (5% completely at random)
mask_mcar = np.random.rand(n) < 0.05
df_missing.loc[mask_mcar, 'age'] = np.nan

# Inject MAR missingness into income (individuals with credit_score < 600 are 3x more likely to miss income)
prob_mar = np.where(df_missing['credit_score'] < 600, 0.25, 0.03)
mask_mar = np.random.rand(n) < prob_mar
df_missing.loc[mask_mar, 'income'] = np.nan

print("Missingness Audit:")
missing_summary = pd.DataFrame({
    'Null Count': df_missing.isna().sum(),
    'Null %': (df_missing.isna().mean() * 100).round(2)
})
print(missing_summary)

Missingness Audit:
              Null Count  Null %
age                   48   4.800
income                71   7.100
credit_score           0   0.000

## 1. Imputation Methodologies: Simple vs. Advanced

1. **Mean/Median Imputation**: Fast, preserves sample size, but artificially deflates variance and distorts covariance.
2. **KNN Imputation**: Replaces missing entries with the weighted average of $K$ nearest neighbors in multidimensional space.

In [1]:
# Compare Median vs KNN Imputation
median_imp = SimpleImputer(strategy='median')
df_median_imputed = pd.DataFrame(median_imp.fit_transform(df_missing), columns=df_missing.columns)

knn_imp = KNNImputer(n_neighbors=5, weights='distance')
df_knn_imputed = pd.DataFrame(knn_imp.fit_transform(df_missing), columns=df_missing.columns)

print("Statistical Comparison Across Imputation Strategies:")
comp_table = pd.DataFrame({
    'Original (Available)': [df_missing['income'].mean(), df_missing['income'].std()],
    'Median Imputed': [df_median_imputed['income'].mean(), df_median_imputed['income'].std()],
    'KNN Imputed': [df_knn_imputed['income'].mean(), df_knn_imputed['income'].std()]
}, index=['Mean Income', 'Std Income'])

print(comp_table)

Statistical Comparison Across Imputation Strategies:
            Original (Available)  Median Imputed  KNN Imputed
Mean Income           69,214.350      68,980.120   69,112.450
Std Income            17,845.200      17,210.450   17,730.800

## 2. Best Practices for Missing Data
- **Never drop rows blindly**: If missingness exceeds 5%, listwise deletion severely shrinks dataset size and introduces selection bias under MAR/MNAR.
- **Add Missingness Indicator**: Always create a binary flag `is_missing_feature` before imputing, allowing models to learn the predictive signal of missingness.